# exp_009 — Phase 1B: Resample length-clipped prompts at 8192 tokens

**Purpose:** Phase 1 found 261 prompts where ALL 4 completions hit the 4096 token cap → registered as uniform-0 (all wrong) but really just truncated. Re-sample those 261 with `max_tokens=8192`. Many will become solvable and shift to sweet-spot.

**Critical change from Phase 1:** **NO judging in this notebook.** Judger has no timeout and hangs on pathological LaTeX. Just dump raw vLLM outputs to a JSONL. Score locally on Mac afterwards using `scripts/score_raw_outputs.py`.

**Inputs needed:**
- `public.jsonl`
- `length_clipped_ids.json` (the 261 IDs to resample)

**Output:** `/content/raw_outputs_long.jsonl` — download and score locally.

**Estimated runtime:** ~45-60 min on A100 40GB.

In [ ]:
# Cell 1 — Install vLLM
# After this cell: Runtime → Restart session, then run from Cell 2.
!pip install -q vllm
print("\nInstalls done. Runtime → Restart session, then run from Cell 2.")

In [ ]:
# Cell 2 — Config
MODEL_NAME = "Qwen/Qwen3-4B-Thinking-2507"

# Sampling
N_SAMPLES_PER_PROMPT = 4
MAX_PROMPT_LEN       = 1024
MAX_COMPLETION_LEN   = 8192   # 2x Phase 1 — recover length-clipped prompts
TEMPERATURE          = 1.0
TOP_P                = 0.95
TOP_K                = 20

# vLLM engine — sized for max_completion=8192 on A100 40GB.
# Math: 16 seqs × 9216 tokens × 147KB/token (Qwen3-4B GQA KV) = ~21 GB KV
# Plus 8 GB model weights = ~29 GB. Fits at gpu_util=0.90 (36 GB allowed).
VLLM_MAX_MODEL_LEN   = 9216    # 1024 prompt + 8192 completion
VLLM_MAX_NUM_SEQS    = 16      # halved from Phase 1 to fit 2x KV cache
VLLM_GPU_UTIL        = 0.90

PILOT_MODE = True
PILOT_N    = 8         # quick smoke test before full 261 prompt run

RANDOM_SEED  = 42
OUTPUT_PATH  = "/content/raw_outputs_long.jsonl"

print(f"PILOT_MODE: {PILOT_MODE}  ({'8 prompts' if PILOT_MODE else 'all 261 prompts'})")
print(f"max_completion: {MAX_COMPLETION_LEN}")
print(f"Model: {MODEL_NAME}")

In [ ]:
# Cell 3 — Upload files
import os, json
from google.colab import files

REQUIRED = [
    ("/content/public.jsonl",          "data/public.jsonl"),
    ("/content/length_clipped_ids.json", "experiments/exp_009_grpo/length_clipped_ids.json"),
]
for path, label in REQUIRED:
    if not os.path.exists(path):
        print(f"Upload: {label}")
        files.upload()
    else:
        print(f"Already present: {path}")

with open("/content/length_clipped_ids.json") as f:
    _data = json.load(f)
TARGET_IDS = set(_data["ids"])
print(f"\nLoaded {len(TARGET_IDS)} target IDs to resample")
print(f"  MCQ:       {_data.get('mcq', '?')}")
print(f"  Free-form: {_data.get('free_form', '?')}")

In [ ]:
# Cell 4 — Prompts (mirror Phase 1 / training notebook)
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

FEWSHOT_MATH = []

FEWSHOT_MCQ = [
    {"role": "user", "content": (
        "Find 1 over 6 + 1 over 8.\n\n"
        "Options:\nA. 7 over 24\nB. 2 over 14\nC. 1 over 4\nD. 7 over 48\n"
        "E. 2 over 24\nF. 1 over 14\nG. 1 over 2\nH. 8 over 14\nI. 0.21 Repeating\nJ. 4 over 24"
    )},
    {"role": "assistant", "content": "Common denominator is 24: 1/6 = 4/24 and 1/8 = 3/24, so 4/24 + 3/24 = 7/24. \\boxed{A}"},
    {"role": "user", "content": (
        "The function value of $\\cos(\\pi + 5i)$ is ( ).\n\n"
        "Options:\nA. -cosh5\nB. -sinh5\nC. sin5i\nD. -sin5\nE. cos5\n"
        "F. cosh5i\nG. sinh5\nH. -cos5\nI. cosh5\nJ. -sin5i"
    )},
    {"role": "assistant", "content": "$\\cos(\\pi + 5i) = -\\cos(5i) = -\\cosh(5)$, using $\\cos(\\pi+x) = -\\cos x$ and $\\cos(ix) = \\cosh x$. \\boxed{A}"},
    {"role": "user", "content": (
        "Find the range of $f(x) = \\frac{ x }{ 1+x^2 }$.\n\n"
        "Options:\nA. -\\frac{1}{3} \\le y \\le \\frac{1}{3}\nB. -\\frac{1}{\\sqrt{3}} \\le y \\le \\frac{1}{\\sqrt{3}}\n"
        "C. -\\frac{1}{4} \\le y \\le \\frac{1}{4}\nD. -\\frac{1}{\\sqrt{2}} \\le y \\le \\frac{1}{\\sqrt{2}}\n"
        "E. -\\frac{1}{\\sqrt{6}} \\le y \\le \\frac{1}{\\sqrt{6}}\nF. -\\frac{1}{2} \\le y \\le \\frac{1}{2}\n"
        "G. -\\frac{1}{\\sqrt{5}} \\le y \\le \\frac{1}{\\sqrt{5}}\nH. -1 \\le y \\le 1\n"
        "I. -\\frac{1}{\\sqrt{7}} \\le y \\le \\frac{1}{\\sqrt{7}}\nJ. -\\frac{1}{\\sqrt{4}} \\le y \\le \\frac{1}{\\sqrt{4}}"
    )},
    {"role": "assistant", "content": "$f'(x) = \\frac{1-x^2}{(1+x^2)^2} = 0$ at $x=\\pm 1$, giving $f(\\pm 1) = \\pm 1/2$. So range is $-1/2 \\le y \\le 1/2$. \\boxed{F}"},
]

In [ ]:
# Cell 5 — Build prompt list, FILTERED to the target 261 IDs
import json

LETTERS = "ABCDEFGHIJ"

prompts = []
with open("/content/public.jsonl") as f:
    for line in f:
        ex = json.loads(line)
        if ex["id"] not in TARGET_IDS:
            continue
        is_mcq = bool("options" in ex and ex["options"])
        question_text = ex["question"]
        if is_mcq:
            opts = "\n".join(f"{LETTERS[i]}. {v}" for i, v in enumerate(ex["options"]))
            question_text = f"{question_text}\n\nOptions:\n{opts}"
            sys_prompt = SYSTEM_PROMPT_MCQ
            fewshots = FEWSHOT_MCQ
        else:
            sys_prompt = SYSTEM_PROMPT_MATH
            fewshots = FEWSHOT_MATH

        msgs = [{"role": "system", "content": sys_prompt}]
        msgs.extend(fewshots)
        msgs.append({"role": "user", "content": question_text})

        prompts.append({
            "id": ex["id"],
            "msgs": msgs,
            "answer": ex["answer"],
            "options": ex.get("options", []),
            "is_mcq": is_mcq,
        })

if PILOT_MODE:
    prompts = prompts[:PILOT_N]
    print(f"PILOT_MODE — {len(prompts)} prompts")
else:
    print(f"Full mode — {len(prompts)} prompts")

print("MCQ:", sum(p["is_mcq"] for p in prompts), "  Free-form:", sum(not p["is_mcq"] for p in prompts))

In [ ]:
# Cell 6 — Initialize vLLM (with Colab patches; see CLAUDE.md / colab_dev.ipynb)
#
# vLLM crashes on Colab because its `suppress_stdout` calls sys.stdout.fileno()
# and ipykernel's iostream raises UnsupportedOperation. We need to give vLLM
# something with .fileno(), but we ALSO need print() to keep reaching Colab's UI.
# The naive fix `open(_fd, 'w', closefd=False)` provides fileno but bypasses
# ipykernel → no prints visible. Use a proxy class that does both.
import os, sys, io as _io

os.environ["VLLM_USE_V1"]                    = "0"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["PYTORCH_ALLOC_CONF"]             = "expandable_segments:True"

class _StdProxy:
    """sys.stdout/stderr proxy: forwards write/flush to original (ipykernel)
    so prints show up in Colab UI, but reports a real OS fd via fileno()
    so vLLM's suppress_stdout doesn't crash."""
    def __init__(self, original, fd):
        self._original = original
        self._fd = fd
    def fileno(self):           return self._fd
    def write(self, s):         return self._original.write(s)
    def flush(self):            return self._original.flush()
    def isatty(self):           return False
    def writable(self):         return True
    @property
    def closed(self):           return False
    def __getattr__(self, name): return getattr(self._original, name)

for _name, _fd in [("stdout", 1), ("stderr", 2)]:
    s = getattr(sys, _name)
    try:
        s.fileno()
    except _io.UnsupportedOperation:
        setattr(sys, _name, _StdProxy(s, _fd))

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

rendered_prompts = [
    tokenizer.apply_chat_template(p["msgs"], tokenize=False, add_generation_prompt=True)
    for p in prompts
]
print(f"Rendered {len(rendered_prompts)} prompts.")
print("Last 200 chars of first prompt:", repr(rendered_prompts[0][-200:]))

print("\nLoading vLLM engine (~2-3 min)...")
llm = LLM(
    model=MODEL_NAME,
    dtype="float16",
    max_model_len=VLLM_MAX_MODEL_LEN,
    max_num_seqs=VLLM_MAX_NUM_SEQS,
    gpu_memory_utilization=VLLM_GPU_UTIL,
    trust_remote_code=True,
    enable_prefix_caching=False,
    seed=RANDOM_SEED,
)

sampling_params = SamplingParams(
    n=N_SAMPLES_PER_PROMPT,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_COMPLETION_LEN,
    seed=RANDOM_SEED,
)
print("vLLM ready.")

In [ ]:
# Cell 7 — Generate and DUMP RAW (no judging — score locally to avoid hangs)
#
# Output schema (per line):
#   {"id": int, "is_mcq": bool, "answer": ..., "options": [...],
#    "completion_texts": [str, ...], "completion_clipped": [bool, ...]}
import time, json

t0 = time.time()
outputs = llm.generate(rendered_prompts, sampling_params)
gen_time = time.time() - t0

print(f"Generation done in {gen_time/60:.1f} min ({gen_time/len(prompts):.1f} sec/prompt average)")
print("Writing raw outputs...")

with open(OUTPUT_PATH, "w") as f:
    for meta, out in zip(prompts, outputs):
        f.write(json.dumps({
            "id": meta["id"],
            "is_mcq": meta["is_mcq"],
            "answer": meta["answer"],
            "options": meta["options"],
            "completion_texts":   [s.text for s in out.outputs],
            "completion_clipped": [len(s.token_ids) >= MAX_COMPLETION_LEN for s in out.outputs],
        }) + "\n")

# Shell commands as defense-in-depth; should be redundant with prints now.
import os
os.system(f"wc -l {OUTPUT_PATH}")
os.system(f"ls -la {OUTPUT_PATH}")
print(f"\nDONE — download {OUTPUT_PATH} via Files panel")